In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

def parse_sentence(sentence):
    doc = nlp(sentence)
    for token in doc:
        print(f"{token.text:15} | dep: {token.dep_:12} | pos: {token.pos_:8} | head: {token.head.text}")
    return doc

doc = parse_sentence("The cat that was sitting on the mat looked very hungry yesterday.")


The             | dep: det          | pos: DET      | head: cat
cat             | dep: nsubj        | pos: NOUN     | head: looked
that            | dep: nsubj        | pos: PRON     | head: sitting
was             | dep: aux          | pos: AUX      | head: sitting
sitting         | dep: relcl        | pos: VERB     | head: cat
on              | dep: prep         | pos: ADP      | head: sitting
the             | dep: det          | pos: DET      | head: mat
mat             | dep: pobj         | pos: NOUN     | head: on
looked          | dep: ROOT         | pos: VERB     | head: looked
very            | dep: advmod       | pos: ADV      | head: hungry
hungry          | dep: acomp        | pos: ADJ      | head: looked
yesterday       | dep: npadvmod     | pos: NOUN     | head: looked
.               | dep: punct        | pos: PUNCT    | head: looked


In [2]:
# Tags we want to REMOVE during compression
REMOVE_DEPS = {"relcl", "prep", "advmod", "npadvmod", "advcl", "appos", "pobj", "amod"}

def compress_sentence(sentence):
    doc = nlp(sentence)
    
    # Find tokens to remove
    remove_indices = set()
    
    def mark_subtree(token):
        remove_indices.add(token.i)
        for child in token.children:
            mark_subtree(child)
    
    for token in doc:
        if token.dep_ in REMOVE_DEPS:
            mark_subtree(token)
    
    # Rebuild sentence with remaining tokens
    compressed = " ".join([token.text for token in doc if token.i not in remove_indices])
    
    # Fix spacing around punctuation
    compressed = compressed.replace(" .", ".").replace(" ,", ",").replace(" '", "'")
    
    return compressed

# Test it
original = "The cat that was sitting on the mat looked very hungry yesterday."
compressed = compress_sentence(original)

print(f"Original:   {original}")
print(f"Compressed: {compressed}")
print(f"Ratio:      {len(compressed.split()) / len(original.split()):.2f}")

Original:   The cat that was sitting on the mat looked very hungry yesterday.
Compressed: The cat looked hungry.
Ratio:      0.33


In [4]:
import sys
!{sys.executable} -m pip install datasets huggingface_hub

  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     -------------- ----------------------- 30.7/82.2 kB 660.6 kB/s eta 0:00:01
     ------------------ ------------------- 41.0/82.2 kB 653.6 kB/s eta 0:00:01
     -------------------------------------  81.9/82.2 kB 573.4 kB/s eta 0:00:01
     -------------------------------------- 82.2/82.2 kB 510.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
    --------------------------------------- 10.2/527.0 kB ? eta -:--:--
    --------------------------------------- 10.2/527.0 kB ? eta -:--:--
   -- ------------------------------------ 30.7


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import importlib
import datasets
importlib.reload(datasets)

<module 'datasets' from 'C:\\Users\\deept\\sentence-compression\\venv\\Lib\\site-packages\\datasets\\__init__.py'>

In [6]:
import sys
print(sys.executable)  # shows which Python Jupyter is using
import datasets
print("datasets version:", datasets.__version__)  # should print a version number

C:\Users\deept\sentence-compression\venv\Scripts\python.exe
datasets version: 4.8.4


In [8]:
import sys
!{sys.executable} -m pip install requests


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import urllib.request
import os

os.makedirs("data", exist_ok=True)

url = "https://raw.githubusercontent.com/google-research-datasets/sentence-compression/master/data/comp-data.eval.json"
dest = "data/comp-data.eval.json"

print("Downloading...")
try:
    urllib.request.urlretrieve(url, dest)
    print(f"✓ Saved to {dest}")
    print(f"✓ File size: {os.path.getsize(dest)} bytes")
except Exception as e:
    print(f"✗ Error: {e}")

Downloading...
✗ Error: HTTP Error 404: Not Found


In [13]:
import requests
import gzip
import os

os.makedirs("data", exist_ok=True)

url = "https://github.com/google-research-datasets/sentence-compression/raw/master/data/comp-data.eval.json.gz"
gz_dest = "data/comp-data.eval.json.gz"
json_dest = "data/comp-data.eval.json"

print("Downloading...")
response = requests.get(url, verify=False)

if response.status_code == 200:
    with open(gz_dest, 'wb') as f:
        f.write(response.content)
    print(f"✓ Downloaded: {os.path.getsize(gz_dest)} bytes")

    print("Extracting...")
    with gzip.open(gz_dest, 'rb') as f_in:
        with open(json_dest, 'wb') as f_out:
            f_out.write(f_in.read())
    print(f"✓ Extracted: {os.path.getsize(json_dest)} bytes")
else:
    print(f"✗ Failed with status: {response.status_code}")

Downloading...


C:\Users\deept\sentence-compression\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\deept\sentence-compression\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ Downloaded: 12183133 bytes
Extracting...
✓ Extracted: 173931593 bytes


In [14]:
import json

samples = []
with open("data/comp-data.eval.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f"✓ Total samples: {len(samples)}")

# Preview first sample
s = samples[0]
source = s['graph']['sentence']
compressed = s['compression']['text']

print("\n--- Sample ---")
print("Source:     ", source)
print("Compressed: ", compressed)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [15]:
with open("data/comp-data.eval.json", "r", encoding="utf-8") as f:
    content = f.read(500)  # Read first 500 characters only

print(repr(content))

'{\n  "graph": {\n    "id": "0",\n    "sentence": "Five people have been taken to hospital with minor injuries following a crash on the A17 near Sleaford this morning.",\n    "node": [ {\n      "form": "ROOT",\n      "word": [ {\n        "id": -1,\n        "form": "ROOT",\n        "stem": "ROOT",\n        "tag": "ROOT"\n      } ],\n      "gender": 0,\n      "head_word_index": 0\n    }, {\n      "form": "Five people",\n      "word": [ {\n        "id": 13,\n        "form": "Five",\n        "stem": "five",\n        "ta'


In [16]:
import json

samples = []
with open("data/comp-data.eval.json", "r", encoding="utf-8") as f:
    content = f.read()

# Split by record — each record starts with {"graph"
raw_records = content.split('\n{')
for i, record in enumerate(raw_records):
    try:
        if i > 0:
            record = '{' + record  # re-add the { we split on
        samples.append(json.loads(record))
    except json.JSONDecodeError:
        continue

print(f"✓ Total samples: {len(samples)}")

# Preview first sample
s = samples[0]
source = s['graph']['sentence']
compressed = s['compression']['text']

print("\n--- Sample ---")
print("Source:     ", source)
print("Compressed: ", compressed)

✓ Total samples: 10000

--- Sample ---
Source:      Five people have been taken to hospital with minor injuries following a crash on the A17 near Sleaford this morning.
Compressed:  Five people have been taken to hospital with minor injuries following a crash on the A17 near Sleaford.


In [17]:
import sys
!{sys.executable} -m pip install rouge-score


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import spacy
nlp = spacy.load("en_core_web_sm")

REMOVE_DEPS = {"relcl", "prep", "advmod", "npadvmod", "advcl", "appos", "pobj", "amod"}

def compress_sentence(sentence):
    doc = nlp(sentence)
    remove_indices = set()

    def mark_subtree(token):
        remove_indices.add(token.i)
        for child in token.children:
            mark_subtree(child)

    for token in doc:
        if token.dep_ in REMOVE_DEPS:
            mark_subtree(token)

    compressed = " ".join([token.text for token in doc if token.i not in remove_indices])
    compressed = compressed.replace(" .", ".").replace(" ,", ",").replace(" '", "'")
    return compressed

print("✓ Compression function ready")

✓ Compression function ready


In [19]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

r1_scores, r2_scores, rL_scores, ratios = [], [], [], []

print("Evaluating on 200 samples...")
for s in samples[:200]:
    source = s['graph']['sentence']
    gold = s['compression']['text']
    predicted = compress_sentence(source)

    scores = scorer.score(gold, predicted)
    r1_scores.append(scores['rouge1'].fmeasure)
    r2_scores.append(scores['rouge2'].fmeasure)
    rL_scores.append(scores['rougeL'].fmeasure)
    ratios.append(len(predicted.split()) / len(source.split()))

print(f"\n✓ Evaluation Complete!")
print(f"ROUGE-1:          {sum(r1_scores)/len(r1_scores):.4f}")
print(f"ROUGE-2:          {sum(r2_scores)/len(r2_scores):.4f}")
print(f"ROUGE-L:          {sum(rL_scores)/len(rL_scores):.4f}")
print(f"Avg Compression:  {sum(ratios)/len(ratios):.4f}")

Evaluating on 200 samples...

✓ Evaluation Complete!
ROUGE-1:          0.5905
ROUGE-2:          0.4595
ROUGE-L:          0.5852
Avg Compression:  0.3653


In [20]:
import json

results = {
    "rouge1": round(sum(r1_scores)/len(r1_scores), 4),
    "rouge2": round(sum(r2_scores)/len(r2_scores), 4),
    "rougeL": round(sum(rL_scores)/len(rL_scores), 4),
    "avg_compression_ratio": round(sum(ratios)/len(ratios), 4),
    "samples_evaluated": 200
}

with open("data/evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to data/evaluation_results.json")
print(json.dumps(results, indent=2))

✓ Results saved to data/evaluation_results.json
{
  "rouge1": 0.5905,
  "rouge2": 0.4595,
  "rougeL": 0.5852,
  "avg_compression_ratio": 0.3653,
  "samples_evaluated": 200
}
